In [ ]:
pip install lightgbm

In [1]:
import pandas as pd
import lightgbm
import yaml
import urllib.request
import os
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
path = "dataset.parquet"
if not os.path.exists(path):
    urllib.request.urlretrieve("https://d20at9gifyo7tz.cloudfront.net/dataset.parquet", path)
else:
    print('Dataset exists')

In [2]:
with Path("./workshop-config.yaml").open("r") as f:
    config: dict = yaml.safe_load(f)

feature_keys = config.get("features", [])
label_key = config.get("label")
grouping_key = config.get("grouping_key")
monotone_constraints = config.get("monotone_constraints", {})

columns_to_load = feature_keys + [label_key, grouping_key]

In [3]:
df = pd.read_parquet(
    "./dataset.parquet",
    columns=columns_to_load,
)
df.shape

(60094814, 24)

In [4]:
monotone_constraints_padded = [
    monotone_constraints.get(key, 0) for key in feature_keys
]

print(list(zip(feature_keys, monotone_constraints_padded)))

[('rentalcar_price_per_day', -1), ('rentalcar_rating', 1), ('rentalcar_renter_rating', 1), ('rentalcar_deposit_rating', 1), ('rentalcar_free_miles', 0), ('insurance_excess', 0), ('rentalcar_insurance_liability_min_euro', 0), ('full_additional_insurance', 0), ('premium_checkin_type', 0), ('status_available_flag', 0), ('is_whitelabel', 0), ('is_expensive_country', 0), ('is_electric', 0), ('supplier_general', 1), ('renter_feedback', 1), ('renter_service', 1), ('country_cluster', 0), ('brand_cluster_label', 0), ('hour_sine_encoded', 0), ('hour_cosine_encoded', 0), ('rentalcar_category', 0), ('rentalcar_type', 0)]


In [5]:
# fix grouping_key dtype
df.loc[:, grouping_key] = df[grouping_key].astype(str)

In [6]:
categorical_feature_keys = (
    df[feature_keys].select_dtypes(include=["category"]).columns.tolist()
)

In [7]:
# check whether there are groups with no non-zero labels
if (df.groupby(grouping_key, observed=True)[label_key].sum() <= 1).sum() > 0:
    raise ValueError("Some groups have no positive labels")

In [8]:
train_data, validation_data = train_test_split(
    df, train_size=0.9, random_state=42
)
train_data: pd.DataFrame
validation_data: pd.DataFrame

In [9]:
train_groups = train_data.groupby(grouping_key, observed=True).size().to_numpy()
validation_groups = (
    validation_data.groupby(grouping_key, observed=True).size().to_numpy()
)

In [10]:
train_set = lightgbm.Dataset(
    data=train_data[feature_keys],
    label=train_data[label_key],
    group=train_groups,
)

validation_set = lightgbm.Dataset(
    data=validation_data[feature_keys],
    label=validation_data[label_key],
    group=validation_groups,
)

In [11]:
booster = lightgbm.train(
    params={
        "objective": "lambdarank",
        "max_depth": -1,
        "num_leaves": 15,
        "min_data_in_leaf": 30,
        "metric": "ndcg",
        "ndcg_eval_at": [1, 3, 5, 10],
        "subsample_for_bin": 150_000,
        "monotone_constraints": monotone_constraints_padded,
    },
    train_set=train_set,
    valid_sets=[validation_set],
    valid_names=["validation"],
    feature_name=feature_keys,
    categorical_feature=categorical_feature_keys,
    num_boost_round=100,
    callbacks=[
        lightgbm.log_evaluation(period=10),
    ],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.381297 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 522
[LightGBM] [Info] Number of data points in the train set: 54085332, number of used features: 22
[10]	validation's ndcg@1: 0.849959	validation's ndcg@3: 0.89419	validation's ndcg@5: 0.90945	validation's ndcg@10: 0.918923
[20]	validation's ndcg@1: 0.85157	validation's ndcg@3: 0.895493	validation's ndcg@5: 0.910607	validation's ndcg@10: 0.919877
[30]	validation's ndcg@1: 0.852085	validation's ndcg@3: 0.895977	validation's ndcg@5: 0.911015	validation's ndcg@10: 0.920265
[40]	validation's ndcg@1: 0.85257	validation's ndcg@3: 0.896316	validation's ndcg@5: 0.911323	validation's ndcg@10: 0.920526
[50]	validation's ndcg@1: 0.852752	validation's ndcg@3: 0.896456	validation's ndcg@5: 0.911412	validation's ndcg@10: 0.920614
[60]	validation's n